In [1]:
import polars as pl

data = pl.read_csv("llm-march2026-alfabank/train_dataset.tsv", separator="\t")

print(data.columns, end="\n\n")
print(data.head(5))

['text', 'target', 'entity']

shape: (5, 3)
┌─────────────────────────────────┬──────────────────────────┬─────────────────────────────────┐
│ text                            ┆ target                   ┆ entity                          │
│ ---                             ┆ ---                      ┆ ---                             │
│ str                             ┆ str                      ┆ str                             │
╞═════════════════════════════════╪══════════════════════════╪═════════════════════════════════╡
│ Вы можете обновить client secr… ┆ []                       ┆ empty                           │
│ Возможно, произошел временный … ┆ []                       ┆ empty                           │
│ Наши запросы с API ключом bk_a… ┆ [(26, 69, 'API ключи')]  ┆ ['bk_api_key_ABCDEF1234567890A… │
│ Сгенерируйте новый JWT для сер… ┆ []                       ┆ empty                           │
│ Возможно, токен 6363069502:y-3… ┆ [(16, 100, 'API ключи')] ┆ ['6363069502:y-3-y-6

In [3]:
# Извлекаем метки из target средствами Polars
# target содержит строки вида "[(26, 69, 'API ключи')]" — извлекаем только названия меток
labels = (
    data
    .filter(pl.col("target") != "[]")
    .select(
        pl.col("target")
        .str.extract_all(r"'([^']+)'")
        .explode()
        .alias("label")
    )
)

# Распределение меток
label_counts = (
    labels
    .group_by("label")
    .len()
    .sort("len", descending=True)
)

print(f"Всего уникальных меток: {label_counts.shape[0]}\n")
print("Распределение меток:")
print(label_counts)

# Строки без меток
empty_count = data.filter(pl.col("target") == "[]").shape[0]
print(f"\nСтрок без меток: {empty_count} из {data.shape[0]} ({empty_count/data.shape[0]*100:.1f}%)")


Всего уникальных меток: 30

Распределение меток:
shape: (30, 2)
┌─────────────────────────────────┬─────┐
│ label                           ┆ len │
│ ---                             ┆ --- │
│ str                             ┆ u32 │
╞═════════════════════════════════╪═════╡
│ 'Полный адрес'                  ┆ 811 │
│ 'Дата рождения'                 ┆ 362 │
│ 'API ключи'                     ┆ 337 │
│ 'Гражданство и названия стран'  ┆ 333 │
│ 'Место рождения'                ┆ 324 │
│ …                               ┆ …   │
│ 'Одноразовые коды'              ┆ 139 │
│ 'Свидетельство о рождении'      ┆ 138 │
│ 'СНИЛС клиента'                 ┆ 133 │
│ 'Email'                         ┆ 129 │
│ 'Серия и номер вида на жительс… ┆ 104 │
└─────────────────────────────────┴─────┘

Строк без меток: 2586 из 8287 (31.2%)


In [ ]:
import re

# Маппинг метка -> int (strip кавычек, т.к. extract_all возвращает полный матч)
unique_labels = sorted(lb.strip("'") for lb in label_counts["label"].to_list())
label2id = {label: i for i, label in enumerate(unique_labels)}

pattern = re.compile(r"\((\d+),\s*(\d+),\s*'([^']+)'\)")

data = data.with_columns(
    pl.col("target").map_elements(
        lambda s: [[int(a), int(b), label2id[c]] for a, b, c in pattern.findall(s)],
        return_dtype=pl.List(pl.List(pl.Int64))
    ).alias("target")
)

print(data.select("text", "target").head(10))

shape: (10, 2)
┌─────────────────────────────────┬─────────────────┐
│ text                            ┆ target          │
│ ---                             ┆ ---             │
│ str                             ┆ list[list[i64]] │
╞═════════════════════════════════╪═════════════════╡
│ Вы можете обновить client secr… ┆ []              │
│ Возможно, произошел временный … ┆ []              │
│ Наши запросы с API ключом bk_a… ┆ [[26, 69, 0]]   │
│ Сгенерируйте новый JWT для сер… ┆ []              │
│ Возможно, токен 6363069502:y-3… ┆ [[16, 100, 0]]  │
│ Мне нужен API ключ для интегра… ┆ []              │
│ Пожалуйста, попробуйте переген… ┆ []              │
│ Наш внутренний бот перестал от… ┆ [[44, 90, 0]]   │
│ Убедитесь, что для Google Clou… ┆ [[40, 122, 0]]  │
│ Я не могу получить выписку по … ┆ [[57, 81, 0]]   │
└─────────────────────────────────┴─────────────────┘


In [16]:
id2label = {v: k for k, v in label2id.items()}

label_dist = (
    data.select(pl.col("target").explode().alias("span"))
    .filter(pl.col("span").is_not_null())
    .select(pl.col("span").list.get(2).alias("label_id"))
    .group_by("label_id")
    .len()
    .sort("len", descending=True)
    .with_columns(pl.col("label_id").replace_strict(id2label).alias("label"))
    .select("label_id", "label", "len")
)

print(f"Уникальных меток: {label_dist.shape[0]}")
print(f"Строк без меток: {data['target'].is_null().sum()}\n")
print(label_dist)


Уникальных меток: 30
Строк без меток: 0

shape: (30, 3)
┌──────────┬─────────────────────────────────┬─────┐
│ label_id ┆ label                           ┆ len │
│ ---      ┆ ---                             ┆ --- │
│ i64      ┆ str                             ┆ u32 │
╞══════════╪═════════════════════════════════╪═════╡
│ 22       ┆ Полный адрес                    ┆ 811 │
│ 10       ┆ Дата рождения                   ┆ 362 │
│ 0        ┆ API ключи                       ┆ 337 │
│ 5        ┆ Гражданство и названия стран    ┆ 333 │
│ 13       ┆ Место рождения                  ┆ 324 │
│ …        ┆ …                               ┆ …   │
│ 18       ┆ Одноразовые коды                ┆ 139 │
│ 26       ┆ Свидетельство о рождении        ┆ 138 │
│ 24       ┆ СНИЛС клиента                   ┆ 133 │
│ 2        ┆ Email                           ┆ 129 │
│ 27       ┆ Серия и номер вида на жительст… ┆ 104 │
└──────────┴─────────────────────────────────┴─────┘


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# ── Config ───────────────────────────────────────────────────────────────────
MODEL_NAME = "masterkristall/rumodernbert_ner_ft_small"
MAX_LENGTH  = 512
BATCH_SIZE  = 16
EPOCHS      = 3
LR          = 5e-5

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# ── BIO label set ────────────────────────────────────────────────────────────
# unique_labels отсортированы → O=0, B-API ключи=1, I-API ключи=2, ...
bio_labels   = ["O"] + [f"{p}-{lbl}" for lbl in unique_labels for p in ("B", "I")]
bio_label2id = {l: i for i, l in enumerate(bio_labels)}
num_ner_labels = len(bio_labels)

print(f"device: {DEVICE} | NER labels: {num_ner_labels}")


# ── Dataset ──────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    def __init__(self, texts, targets):
        self.texts   = texts
        self.targets = [t if t else [] for t in targets]

    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.targets[idx]


# ── Collate ──────────────────────────────────────────────────────────────────
def collate_fn(batch, tokenizer):
    texts, targets = zip(*batch)

    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
        return_offsets_mapping=True,
    )
    offset_mappings = encoded.pop("offset_mapping")

    token_labels_batch, binary_labels_batch = [], []

    for spans, offsets in zip(targets, offset_mappings):
        binary_labels_batch.append(1 if spans else 0)
        token_labels = []
        for tok_start, tok_end in offsets.tolist():
            if tok_start == tok_end:          # [CLS] / [SEP] / padding
                token_labels.append(-100)
                continue
            label = bio_label2id["O"]
            for span_start, span_end, label_id in spans:
                if span_start <= tok_start < span_end:
                    name  = unique_labels[label_id]
                    label = bio_label2id[f"B-{name}" if tok_start == span_start else f"I-{name}"]
                    break
            token_labels.append(label)
        token_labels_batch.append(token_labels)

    encoded["labels"]        = torch.tensor(token_labels_batch,  dtype=torch.long)
    encoded["binary_labels"] = torch.tensor(binary_labels_batch, dtype=torch.long)
    return encoded


# ── Model ─────────────────────────────────────────────────────────────────────
class MultiTaskNER(nn.Module):
    def __init__(self, model_name, num_ner_labels):
        super().__init__()
        self.encoder  = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.ner_head = nn.Linear(h, num_ner_labels)   # токен → класс NER
        self.cls_head = nn.Linear(h, 2)                # CLS  → есть ли ПД

    def forward(self, input_ids, attention_mask, labels=None, binary_labels=None, **kwargs):
        hidden      = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        ner_logits  = self.ner_head(hidden)          # (B, T, num_ner_labels)
        cls_logits  = self.cls_head(hidden[:, 0])    # (B, 2)

        loss = None
        if labels is not None:
            ner_loss = nn.CrossEntropyLoss()(ner_logits.view(-1, num_ner_labels), labels.view(-1))
            cls_loss = nn.CrossEntropyLoss()(cls_logits, binary_labels)
            loss = ner_loss + cls_loss

        return {"loss": loss, "ner_logits": ner_logits, "cls_logits": cls_logits}


# ── Data & training ───────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset   = NERDataset(data["text"].to_list(), data["target"].to_list())
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=lambda b: collate_fn(b, tokenizer),
)

model     = MultiTaskNER(MODEL_NAME, num_ner_labels).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(dataloader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for step, batch in enumerate(dataloader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss  = model(**batch)["loss"]

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        if (step + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | step {step+1}/{len(dataloader)} | loss {loss.item():.4f}")

    print(f"── Epoch {epoch+1}/{EPOCHS} avg loss: {total_loss / len(dataloader):.4f}")
